# Experiment 2: Tackle Noise with Error Correction

### Aim:
Tackle noise with error correction using 3-qubit bit-flip code and physically restore the codeword to |111>.

---
### Dataset Required:
None. Error correction operates directly on the quantum states.

---
### Libraries Required & Purpose:
* `qiskit`: To build encoding, error injection, syndrome extraction, correction, and measurement gates.
* `qiskit_aer`: To simulate the circuit and observe error recovery.

---


### Step 1: Import Qiskit and Simulator

In [ ]:
# Import QuantumCircuit and transpile from qiskit
from qiskit import QuantumCircuit, transpile
# Import AerSimulator for local execution
from qiskit_aer import AerSimulator

### Step 2: Build 3-Qubit Error Correction Circuit with Physical Restoration to |111>
- Qubits 0, 1, 2: 3 Physical Data Qubits
- Qubits 3, 4: 2 Syndrome Ancilla Qubits
- Bits 0, 1, 2: 3 Classical Bits to measure the data qubits

In [ ]:
# Create circuit: 3 data qubits (0, 1, 2) + 2 syndrome ancillas (3, 4) and 3 classical bits
qc = QuantumCircuit(5, 3)

# 1. State Preparation: Prepare qubit 0 in state |1>
qc.x(0)

# 2. Encoding: Entangle qubit 0 with qubits 1 and 2 (forms logical repetition code |111>)
qc.cx(0, 1)
qc.cx(0, 2)

# 3. Noise Injection: Introduce a bit-flip (X) error on qubit 1 (state becomes |101>)
qc.x(1)

# 4. Syndrome Extraction: Measure parities using ancillas (qubit 3 and qubit 4)
# Ancilla 3 checks parity (q0 ^ q1)
qc.cx(0, 3)
qc.cx(1, 3)
# Ancilla 4 checks parity (q1 ^ q2)
qc.cx(1, 4)
qc.cx(2, 4)

# 5. Error Correction:
# If syndrome is 11 (both ancillas are 1), error is on qubit 1.
# Apply Toffoli (CCX) using ancillas (3, 4) as controls to flip qubit 1 back to 1!
qc.ccx(3, 4, 1)

# 6. Measurement: Measure all 3 physical data qubits into classical bits (0, 1, 2)
qc.measure([0, 1, 2], [0, 1, 2])

# Display the circuit diagram
print("=== Error Correction Circuit Diagram ===")
print(qc.draw(output='text'))

### Step 3: Simulate and Verify Full Restoration to |111>

In [ ]:
# Run the circuit on the simulator
sim = AerSimulator()
compiled_qc = transpile(qc, sim)
result = sim.run(compiled_qc, shots=1024).result()
counts = result.get_counts()

# The measurement counts will be exactly '111' in 100% of shots!
print("Measurement counts of the 3 physical qubits:", counts)
print("Physical state successfully restored to |111>!")